## Configuration


In [ ]:
dbutils.widgets.text("catalog", "training_lab")
dbutils.widgets.text("schema", "student_01")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema)
ns = f"{catalog}.{schema}"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
print("Working namespace:", ns)
from pyspark.sql import functions as F


## Create the teaching datasets


In [ ]:
from datetime import date, timedelta
import math, random
orders = [("O1","C1","2026-01-01",100.0),
 ("O2","C2","2026-01-01",200.0),
 ("O3","C1","2026-01-02",150.0),
 ("O4","C3","2026-01-02",400.0),
 ("O5","C2","2026-01-03",250.0),
 ("O6","C3","2026-01-03",300.0)]
o = spark.createDataFrame(orders,
 "order_id string, customer_id string, order_date string, amount double")
o = o.withColumn("order_date", F.to_date("order_date"))
c = spark.createDataFrame([
 ("C1","Asha","Bengaluru"),("C2","Ravi","Hyderabad"),
 ("C3","Meena","Chennai"),("C4","Imran","Pune")],
 "customer_id string, name string, city string")
p = spark.createDataFrame([
 ("P1","O1",60.0),("P2","O1",40.0),("P3","O2",200.0),
 ("P4","O3",150.0),("P5","O4",400.0),("P6","O5",250.0)],
 "payment_id string, order_id string, paid double")
for name, df in [("orders",o),("customers",c),("payments",p)]:
 df.write.format("delta").mode("overwrite").saveAsTable(ns+".day18_"+name)
rng = random.Random(1801)
daily = []
for i in range(140):
 d = date(2026,1,1) + timedelta(days=i)
 units = max(0, round(100 + 0.35*i + 18*math.sin(2*math.pi*i/7)
                     + rng.gauss(0,3)))
 daily.append((d, float(units)))
spark.createDataFrame(daily,"ds date, y double").write.format("delta")\
 .mode("overwrite").saveAsTable(ns+".day18_daily_demand")
rng = random.Random(1802)
shipments = []
for i in range(400):
 distance = float(rng.randint(5,150))
 items = float(rng.randint(1,20))
 rain = float(rng.randint(0,1))
 hours = max(0.5, 2 + 0.055*distance + 0.07*items
             + 1.4*rain + rng.gauss(0,0.8))
 shipments.append((i,distance,items,rain,float(hours),float(hours>8)))
spark.createDataFrame(shipments,
 "shipment_id int, distance_km double, items double, rain double, "
 "delivery_hours double, late double").write.format("delta")\
 .mode("overwrite").saveAsTable(ns+".day18_shipments")
assert o.count() == 6 and c.count() == 4 and p.count() == 6
assert o.agg(F.sum("amount")).first()[0] == 1400.0
assert spark.table(ns+".day18_daily_demand").count() == 140
assert spark.table(ns+".day18_shipments").count() == 400
print("Setup passed: 6 orders, 4 customers, 6 payments, 140 days, 400 shipments")
